# 04 Neurosymbolic Inference Heatmap

Notebook demo final untuk inference Neuro-Symbolic dari gambar upload.

- Faster R-CNN tetap menyediakan backbone, RPN, RoI Align, bbox regressor, dan post-processing deteksi.
- SODT menggantikan cabang klasifikasi dan menerima flattened RoI Align pooled grid `[C, 7, 7]`.
- Explanation yang ditampilkan adalah **Local Evidence per decision node**, bukan heatmap path gabungan.


In [1]:
from pathlib import Path
import io

import ipywidgets as widgets
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import clear_output, display
from PIL import Image

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.preprocess_dataset import test_preprocess
from neurosym.inference import (
    explain_hybrid_detection,
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    select_detection_indices,
)
from util.artifacts import latest_run_checkpoint
from util.config import load_yaml
from util.device import select_device

PROJECT_ROOT = resolve_root()

In [2]:
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)

device = select_device(train_config["device"])
detector_checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / "checkpoints" / "neuro")
symbolic_checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / "checkpoints" / "symbolic")

hybrid_model, detector_checkpoint = load_neurosymbolic_detector(
    detector_checkpoint_path=detector_checkpoint_path,
    neuro_config=neuro_config,
    train_config=train_config,
    symbolic_checkpoint_path=symbolic_checkpoint_path,
    device=str(device),
)

image_preprocess = test_preprocess()
class_names = tuple(train_config["dataset"]["class_names"])

detector_checkpoint_path, symbolic_checkpoint_path

(PosixPath('/home/wiszel/OneDrive/Documents/WiszeL/UNS/Semester 8/pcb_skripsi/checkpoints/neuro/run1.pt'),
 PosixPath('/home/wiszel/OneDrive/Documents/WiszeL/UNS/Semester 8/pcb_skripsi/checkpoints/symbolic/run3.pt'))

In [3]:
def image_to_array(image_tensor: torch.Tensor) -> np.ndarray:
    return image_tensor.detach().cpu().permute(1, 2, 0).clamp(0.0, 1.0).numpy()


def heatmap_to_array(heatmap: torch.Tensor) -> np.ndarray:
    import matplotlib.pyplot as plt
    import numpy as np
    arr = heatmap.detach().cpu().numpy()
    arr_norm = np.clip(arr, 0.0, 1.0)
    if arr_norm.max() > 0:
        arr_norm = arr_norm / arr_norm.max()
    
    rgba = plt.get_cmap("jet")(arr_norm)
    
    # Remove noise entirely, but make the active area VERY opaque and bright
    # arr_norm ** 2.0 strongly suppresses background noise to yield sharp heatmaps.
    alpha = np.where(arr_norm > 0.15, arr_norm ** 1.0, 0.0)
    rgba[..., 3] = alpha
    return rgba
def class_name(label: int) -> str:
    return class_names[int(label) - 1]


def zoom_axis_to_box(
    axis: plt.Axes,
    box: torch.Tensor,
    image_shape: tuple[int, int],
    padding_ratio: float = 0.2,
    minimum_crop_size: int = 48,
) -> None:
    image_height, image_width = int(image_shape[0]), int(image_shape[1])
    x1, y1, x2, y2 = [float(value) for value in box.detach().cpu().tolist()]
    box_width = max(x2 - x1, 1.0)
    box_height = max(y2 - y1, 1.0)
    crop_width = max(box_width * (1.0 + 2.0 * padding_ratio), float(minimum_crop_size))
    crop_height = max(
        box_height * (1.0 + 2.0 * padding_ratio), float(minimum_crop_size)
    )
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0

    left = max(center_x - crop_width / 2.0, 0.0)
    right = min(center_x + crop_width / 2.0, float(image_width))
    top = max(center_y - crop_height / 2.0, 0.0)
    bottom = min(center_y + crop_height / 2.0, float(image_height))

    axis.set_xlim(left, right)
    axis.set_ylim(bottom, top)


def draw_numbered_detections(
    axis: plt.Axes,
    image_tensor: torch.Tensor,
    detection_result: dict[str, torch.Tensor],
    detection_indices: list[int],
    selected_index: int | None = None,
    display_numbers: list[int] | None = None,
) -> None:
    axis.imshow(image_to_array(image_tensor))
    # Add a separate black dimmer layer over the image so it doesn't corrupt the heatmap colors
    dimmer = np.zeros((image_tensor.shape[-2], image_tensor.shape[-1], 4), dtype=np.float32)
    dimmer[..., 3] = 0.35  # 40% perfect black dimmer
    axis.imshow(dimmer)
    axis.axis("off")

    if display_numbers is None:
        display_numbers = list(range(1, len(detection_indices) + 1))

    for display_number, detection_index in zip(display_numbers, detection_indices):
        box = detection_result["boxes"][detection_index].detach().cpu()
        label = int(detection_result["labels"][detection_index])
        score = float(detection_result["scores"][detection_index])
        x1, y1, x2, y2 = box.tolist()
        edge_color = "lime" if detection_index == selected_index else "red"
        line_width = 3 if detection_index == selected_index else 2
        axis.add_patch(
            patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=line_width,
                edgecolor=edge_color,
                facecolor="none",
                clip_on=True,
            )
        )
        axis.text(
            x1,
            max(y1 - 5, 0),
            f"#{display_number} {class_name(label)} {score:.2f}",
            color="black",
            fontsize=9,
            weight="bold",
            clip_on=True,
            bbox={"facecolor": "yellow", "edgecolor": edge_color, "pad": 2},
        )

In [ ]:
upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image",
)
MAX_DISPLAY_DETECTIONS = 9
DISPLAY_SCORE_THRESHOLD = 0.3
detection_output = widgets.Output()
explanation_output = widgets.Output()
state = {
    "image_name": None,
    "image_tensor": None,
    "detection": None,
    "selected_indices": [],
}


def uploaded_file_record():
    value = upload_widget.value
    if isinstance(value, tuple):
        return value[0] if value else None
    if isinstance(value, dict):
        if "content" in value:
            return value
        return next(iter(value.values())) if value else None
    return None


def uploaded_content_bytes(file_record) -> bytes:
    content = file_record["content"]
    return content.tobytes() if isinstance(content, memoryview) else bytes(content)


def render_detection(detection_index: int) -> None:
    image_tensor = state["image_tensor"]
    detection = state["detection"]
    selected_indices = state["selected_indices"]
    explanation = explain_hybrid_detection(
        hybrid_model,
        detection,
        detection_index=detection_index,
        image_shape=tuple(image_tensor.shape[-2:]),
    )
    label_name = class_name(explanation["label"])

    with explanation_output:
        clear_output(wait=True)
        fig, axis = plt.subplots(figsize=(8, 8))
        selected_number = selected_indices.index(detection_index) + 1
        draw_numbered_detections(
            axis,
            image_tensor,
            detection,
            [detection_index],
            selected_index=detection_index,
            display_numbers=[selected_number],
        )
        zoom_axis_to_box(
            axis, explanation["detection_box"], tuple(image_tensor.shape[-2:])
        )
        axis.set_title(
            f"Zoomed detection #{selected_number}: {label_name} {explanation['score']:.2f}"
        )
        plt.show()

        import matplotlib.gridspec as gridspec
        node_count = len(explanation["node_explanations"])
        fig = plt.figure(figsize=(13, max(8, 4 * node_count)))
        gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1], wspace=-0.18)
        
        # 1. LEFT SIDE: Pruned SODT Tree
        ax_tree = plt.subplot(gs[0])
        ax_tree.axis("off")
        ax_tree.set_title("Pruned SODT Tree", fontsize=16, weight="bold")
        
        active_nodes = {n["node_index"]: n for n in explanation["node_explanations"]}
        tree_depth = hybrid_model.symbolic_tree.max_depth
        num_internal = (2 ** tree_depth) - 1
        num_leaves = 2 ** tree_depth
        leaf_node = explanation["symbolic_leaf_index"] + num_internal
        
        # Identify structurally pruned internal nodes (weights and bias are zero)
        pruned_nodes = set()
        for idx in range(num_internal):
            if np.all(hybrid_model.symbolic_tree.node_weights[idx] == 0.0) and hybrid_model.symbolic_tree.node_bias[idx] == 0.0:
                pruned_nodes.add(idx)
                
        # Build the structurally visible pruned tree
        visible_nodes = set()
        visible_edges = []
        leaf_override = {}
        
        def build_pruned_tree(node, depth):
            visible_nodes.add(node)
            if node >= num_internal:
                return
            if node in pruned_nodes:
                # Find leftmost leaf class index in the pure collapsed subtree
                curr = node
                while curr < num_internal:
                    curr = curr * 2 + 1
                leaf_offset = curr - num_internal
                label_idx = hybrid_model.symbolic_tree.leaf_labels[leaf_offset]
                leaf_override[node] = class_name(label_idx)
                return
            
            left = node * 2 + 1
            right = node * 2 + 2
            visible_edges.append((node, left, "left"))
            visible_edges.append((node, right, "right"))
            build_pruned_tree(left, depth + 1)
            build_pruned_tree(right, depth + 1)
            
        build_pruned_tree(0, 0)
        
        # --- NEW PRETTY LAYOUT ALGORITHM ---
        # 1. Build parent-child relationships
        children_map = {node: [] for node in visible_nodes}
        for p, c, _ in visible_edges:
            children_map[p].append(c)
        
        coords = {}
        leaf_x_counter = [0]
        
        def assign_coords(node, depth):
            # Sort children to maintain left-to-right order (left child index < right child index)
            children = sorted(children_map[node])
            if not children:
                # Terminal node (spaced evenly)
                coords[node] = (leaf_x_counter[0], -depth * 2.5)
                leaf_x_counter[0] += 40  # Give plenty of horizontal spacing between leaves
            else:
                child_x_sum = 0
                for c in children:
                    assign_coords(c, depth + 1)
                    child_x_sum += coords[c][0]
                coords[node] = (child_x_sum / len(children), -depth * 2.5)
                
        assign_coords(0, 0)
        
        # 2. Center the tree around x=0
        min_x_coord = min(x for x, y in coords.values())
        max_x_coord = max(x for x, y in coords.values())
        center_offset = (max_x_coord + min_x_coord) / 2.0
        for node in coords:
            x, y = coords[node]
            coords[node] = (x - center_offset, y)
        
        # Helper to check if ancestor reaches leaf_node
        def is_ancestor(ancestor, descendent):
            if ancestor == descendent:
                return True
            if ancestor >= num_internal:
                return False
            return is_ancestor(ancestor * 2 + 1, descendent) or is_ancestor(ancestor * 2 + 2, descendent)
            
        for parent, child, side in visible_edges:
            is_active_parent = parent in active_nodes
            child_active = is_active_parent and is_ancestor(child, leaf_node)
            
            color = "#388e3c" if (child_active and side == "left") else ("#d32f2f" if (child_active and side == "right") else "#e0e0e0")
            lw = 3 if child_active else 1
            zorder = 2 if child_active else 1
            ax_tree.plot([coords[parent][0], coords[child][0]], [coords[parent][1], coords[child][1]], color=color, lw=lw, zorder=zorder)
            
        for node in visible_nodes:
            x, y = coords[node]
            is_leaf = (node >= num_internal) or (node in leaf_override)
            is_active = (node in active_nodes) or (node == leaf_node) or (node in leaf_override and is_ancestor(node, leaf_node))
            
            alpha = 1.0 if is_active else 0.5  # Slightly more opaque for unselected nodes
            fc = "#e1f5fe" if not is_leaf else "#c8e6c9"
            ec = "black" if is_active else "#9e9e9e"  # Slightly darker grey edge
            if not is_active: fc = "#f5f5f5"
            lw = 2 if is_active else 1
            
            # --- MAKE UNSELECTED NODES BIGGER ---
            internal_font_active = 11
            internal_font_inactive = 9
            leaf_font_active = 11
            leaf_font_inactive = 9
            
            if is_leaf:
                if node in leaf_override:
                    label = leaf_override[node]
                else:
                    label = f"{label_name}" if node == leaf_node else f"L{node - num_internal}"
                
                leaf_active = (node == leaf_node) or (node in leaf_override and is_ancestor(node, leaf_node))
                if leaf_active:
                    fc = "#4caf50"
                    ec = "#1b5e20"
                bbox = dict(boxstyle="round,pad=0.3", fc=fc, ec=ec, lw=lw, alpha=alpha)
                ax_tree.text(x, y, label, ha="center", va="center", fontsize=leaf_font_active if is_active else leaf_font_inactive, weight="bold" if is_active else "normal", bbox=bbox, zorder=3, rotation=90, rotation_mode="anchor")
            else:
                label = f"{active_nodes[node]['score']:.2f}" if node in active_nodes else f"N{node}"
                bbox = dict(boxstyle="circle,pad=0.2", fc=fc, ec=ec, lw=lw, alpha=alpha)
                ax_tree.text(x, y, label, ha="center", va="center", fontsize=internal_font_active if is_active else internal_font_inactive, weight="bold" if is_active else "normal", bbox=bbox, zorder=3)
                
        # Dynamically scale axis limits so the text never overlaps the edge
        min_final_x = min(x for x, y in coords.values())
        max_final_x = max(x for x, y in coords.values())
        ax_tree.set_xlim(min_final_x - 8, max_final_x + 8)
        ax_tree.set_ylim(-tree_depth * 2.5 - 1.0, 1.0)
        
        # 2. RIGHT SIDE: Vertical Heatmaps
        gs_right = gridspec.GridSpecFromSubplotSpec(node_count, 1, subplot_spec=gs[1], hspace=0.4)
        for axis_index, node in enumerate(explanation["node_explanations"]):
            axis = plt.subplot(gs_right[axis_index])
            axis.imshow(image_to_array(image_tensor))
            # Add a separate black dimmer layer over the image so it doesn't corrupt the heatmap colors
            dimmer = np.zeros((image_tensor.shape[-2], image_tensor.shape[-1], 4), dtype=np.float32)
            dimmer[..., 3] = 0.4  # 40% perfect black dimmer
            axis.imshow(dimmer)
            axis.imshow(
                heatmap_to_array(node["projected_node_heatmap_on_detection_box"]),
                cmap="jet",
                vmin=0.0,
                vmax=1.0,
            )
            x1, y1, x2, y2 = explanation["detection_box"].tolist()
            axis.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="cyan",
                    facecolor="none",
                )
            )
            zoom_axis_to_box(
                axis, explanation["detection_box"], tuple(image_tensor.shape[-2:])
            )
            axis.set_title(
                f"Depth {node['depth'] + 1} | Node {node['node_index']} Features\n"
                f"Score: {node['score']:.2f} -> Went {node['decision'].upper()}"
            )
            axis.axis("off")
            
        plt.show()


def make_detection_button(display_number: int, detection_index: int) -> widgets.Button:
    detection = state["detection"]
    label = int(detection["labels"][detection_index])
    score = float(detection["scores"][detection_index])
    button = widgets.Button(
        description=f"#{display_number} {class_name(label)} {score:.2f}",
        layout=widgets.Layout(width="180px"),
    )
    button.on_click(lambda _: render_detection(detection_index))
    return button


def run_uploaded_inference(_button) -> None:
    file_record = uploaded_file_record()
    with detection_output:
        clear_output(wait=True)
        explanation_output.clear_output(wait=True)

        if file_record is None:
            print("Upload one PCB image first.")
            return

        image_name = file_record.get("name", "uploaded_image")
        pil_image = Image.open(io.BytesIO(uploaded_content_bytes(file_record))).convert(
            "RGB"
        )
        image_tensor = image_preprocess(pil_image)
        detection = run_neurosymbolic_inference(hybrid_model, [image_tensor])[0]
        selected_indices = select_detection_indices(
            detection,
            score_threshold=DISPLAY_SCORE_THRESHOLD,
            max_detections=MAX_DISPLAY_DETECTIONS,
        )

        state["image_name"] = image_name
        state["image_tensor"] = image_tensor
        state["detection"] = detection
        state["selected_indices"] = selected_indices

        fig, axis = plt.subplots(figsize=(8, 8))
        draw_numbered_detections(axis, image_tensor, detection, selected_indices)
        axis.set_title(f"Neuro-Symbolic detections: {image_name}")
        plt.show()

        if not selected_indices:
            print("No detections were returned by the model.")
            return

        buttons = [
            make_detection_button(display_number, detection_index)
            for display_number, detection_index in enumerate(selected_indices, start=1)
        ]
        display(
            widgets.GridBox(
                buttons,
                layout=widgets.Layout(
                    grid_template_columns="repeat(3, 190px)",
                    grid_gap="8px",
                ),
            )
        )

        render_detection(selected_indices[0])


def run_uploaded_inference_on_upload(change) -> None:
    if change["new"]:
        run_uploaded_inference(None)


upload_widget.observe(run_uploaded_inference_on_upload, names="value")
display(
    widgets.VBox(
        [
            upload_widget,
            detection_output,
            explanation_output,
        ]
    )
)